> Task2：  
> - 问题的定义：调用标准库 SB3 ，解决稍微困难些的问题 PandaReach-v3
> - 目标：  
> 1. 尝试调用 SB3 ，了解其功能   
> 2. 尝试调用 SB3 处理一个未知环境（例如 PandaReach-v3），并了解 SOP 流程 。如何探索环境？如何调用 SB3 选择算法？如何逐步调试到最终落实？等等  

## Part1: 探索环境

> 问题重述-综下所述：  
PandaReach-v3：  
- Task：  
    - Reach。机械臂需要移动其末端执行器（End-effector，即“手掌”位置），去触碰空间中随机生成的一个红色目标点（Target）。  
    - Challenge：机械臂需要学习三维空间的坐标转换和运动规划，以最短路径到达目标。  
- Robot：  
    - Franka Emika Panda。7DOF（旋转关节）。  
- Action Space：  
    - 3维 (末端执行器X, Y, Z 位移)。Box(-1.0, 1.0, (3,))，连续动作空间。  
    PS：3D 位移指令会通过环境内部的 逆运动学 (Inverse Kinematics, IK) 自动转换为 7 个关节的旋转角度。  
- Observation Space：  
    - 12 (6 obs + 3 achieved + 3 desired)  
- Reward Setting：  
    - 选择Dense Reward。末端执行器与目标点之间 欧几里得距离的负值  
- done 条件：  
    - Success：当末端执行器距离目标的欧氏距离 ≤0.05m (5厘米) 时，认为任务成功。  
    - Truncated：50 步内没能到达目标，该回合结束。  
- 仿真引擎：基于 PyBullet，提供真实的重力、摩擦力及碰撞物理效果  

In [3]:
import gymnasium as gym
import panda_gym

env = gym.make("PandaReach-v3")

print(f"观察空间 (Observation Space): {env.observation_space}")
print(f"动作空间 (Action Space): {env.action_space}")
print(f"动作范围: {env.action_space.high}, {env.action_space.low}")

pybullet build time: Oct 21 2025 17:40:50


argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
观察空间 (Observation Space): Dict('achieved_goal': Box(-10.0, 10.0, (3,), float32), 'desired_goal': Box(-10.0, 10.0, (3,), float32), 'observation': Box(-10.0, 10.0, (6,), float32))
动作空间 (Action Space): Box(-1.0, 1.0, (3,), float32)
动作范围: [1. 1. 1.], [-1. -1. -1.]


> Gemini：  
观察空间是 Dict：包含 observation, desired_goal 等。  
结论：我必须使用 SB3 的 MultiInputPolicy，普通的 MlpPolicy 处理不了字典。  
动作空间是 Box(3,) 且连续：  
结论：排除 DQN（它是离散的）。我需要在 SAC, PPO, DDPG, TD3 中选。  
任务属性：这是机器人控制（Robotics）。  
结论：SAC 是目前的行业首选，因为它的样本效率最高（Off-policy）。  

> PS: 关于物理引擎：通过 gymnasium 接口统一调用 mujoco或者 pybullet。  

> import:  
mujoco：import gymnasium as gym  
pybullet：import gymnasium as gym+import panda_gym  

> 渲染方式：  
gym.make(env_id, render_mode="human") -> 弹出一个物理引擎的 3D 窗口。  
gym.make(env_id, render_mode="rgb_array") -> 不弹窗，只返回图像数据（用于记录视频）。

> 注意observation：  
Panda-Gym (PyBullet)：大多数环境是 Goal-Conditioned (目标导向) 的。它们返回的 observation 是一个 字典 (Dict)，包含 observation、achieved_goal 和 desired_goal。  
你的代码中 SB3 会自动识别这种字典并使用 MultiInputPolicy（如果你没显式指定的话）。  
标准 MuJoCo (如 Ant-v4)：大多数返回的是一个简单的 数组 (Box/Array)。  
这时候 SB3 需要使用 MlpPolicy。  

In [4]:
# Random Baseline 探索一下奖励情况
obs, _ = env.reset()
total_reward = 0
for _ in range(1000):
    action = env.action_space.sample() # 随机采样
    obs, reward, terminated, truncated, _ = env.step(action)
    total_reward += reward
    if terminated or truncated:
        obs, _ = env.reset()
print(f"随机策略平均奖励: {total_reward / 1000}")

随机策略平均奖励: -0.998


> Gemini：  
如果奖励全是 0，说明是稀疏奖励（极难）。  
如果奖励是连续的小负数，说明是密集奖励（较易）。  
PandaReach 结论：它有 dense 模式，我们可以从简单的 dense 开始。  

### experiment_01 SAC最简化实现 & 定义 save 路径

In [ ]:
# SAC 最简化实现 & TensorBoard
import os
from datetime import datetime
import gymnasium as gym
import panda_gym
from stable_baselines3 import SAC


# 1. 定义 save 路径
exp_name = "exp_01_SAC_random"
base_path = f"./outputs/{exp_name}"

best_model_dir = os.path.join(base_path, "best_model")
final_model_dir = os.path.join(base_path, "final_model")
log_dir = os.path.join(base_path, "logs")
os.makedirs(best_model_dir, exist_ok=True)
os.makedirs(final_model_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

env = gym.make("PandaReach-v3")

# 通过 tensorboard_log 自动把 log 保存到 “tensorboard_log/tb_log_name“ 
model = SAC("MultiInputPolicy", env, verbose=0, tensorboard_log=log_dir)
model.learn(total_timesteps=20000, tb_log_name=datetime.now().strftime("%Y%m%d_%H%M")) 

argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886


In [ ]:
'''
# save 路径
exp_name = "exp_01_SAC_random"
base_path = f"./outputs/{exp_name}"

best_model_dir = os.path.join(base_path, "best_model")
final_model_dir = os.path.join(base_path, "final_model")
log_dir = os.path.join(base_path, "logs")
os.makedirs(best_model_dir, exist_ok=True)
os.makedirs(final_model_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

# 具体的模型与统计量文件路径
final_model_path = os.path.join(final_model_dir, "final_model.zip")
final_vec_path = os.path.join(final_model_dir, "final_model_vec_normalize.pkl")
'''

只要你开启了 tensorboard_log，SB3 默认会自动记录以下三类数据（取决于具体算法，SAC 通常包含）：  
rollout/ (评估相关)：  
ep_rew_mean: 过去 100 个 episode 的平均奖励（最重要指标）。  
ep_len_mean: 过去 100 个 episode 的平均步数。  
success_rate: 成功率（如果你的环境提供了 is_success 信息）。  
train/ (训练细节)：  
learning_rate: 当前学习率。  
loss: 总损失。  
actor_loss / critic_loss: 策略和价值网络的各自损失（针对 SAC）。  
ent_coef: 熵系数（反映 SAC 的探索程度）。  
time/ (效率相关)：  
fps: 每秒处理的步数（衡量训练速度）。  
time_elapsed: 训练累计耗时。  


In [ ]:
# 通过 SB3 的 Callback 来记录更多自定义数据
from stable_baselines3.common.callbacks import BaseCallback
'''
class CustomDataCallback(BaseCallback):

my_callback = CustomDataCallback()

model = SAC("MultiInputPolicy", env, tensorboard_log="./logs/")
model.learn(total_timesteps=10000, callback=my_callback)
'''

观察现象：
你会发现奖励（Rew Mean）在动，但提升非常缓慢，或者干脆不动。  
诊断： 检查 TensorBoard，如果你发现输入数据的数值（如坐标）跨度很大（从 0 到 1），而网络权重更新很吃力。  
药方： 必须引入 VecNormalize（归一化）。  

### experiment_02 SAC最简化+DummyVecEnv, VecNormalize

In [ ]:
import os
from datetime import datetime

import gymnasium as gym
import panda_gym

from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize


# 1. 定义 save 路径
exp_name = "exp_02_SAC_random"
base_path = f"./outputs/{exp_name}"

best_model_dir = os.path.join(base_path, "best_model")
final_model_dir = os.path.join(base_path, "final_model")
log_dir = os.path.join(base_path, "logs")
os.makedirs(best_model_dir, exist_ok=True)
os.makedirs(final_model_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

# 2. 定义并包装环境
env_id = "PandaReach-v3"

def make_env():
    # 当前环境设置 reward_type="dense" 确保快速收敛
    env = gym.make(env_id, reward_type="dense")
    return env
# env = make_env()

# VecEnv 需要在一个 lambda 函数中创建
env = DummyVecEnv([make_env])

# 添加 VecNormalize 包装器
# norm_obs: 是否归一化观测值 (通常建议开启)—稳定Actors-Critic网络
# norm_reward: 是否归一化奖励 (通常建议开启)—稳定Critic网络
# clip_obs: 裁剪观测值的范围—防止极端异常值破坏神经网络的权重
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)

# 3. main & tensorboard
# 注意：使用 VecNormalize 后，模型输入的是归一化后的数据
model = SAC(
    "MultiInputPolicy", 
    env, 
    verbose=0, 
    tensorboard_log=log_dir
)
model.learn(
    total_timesteps=20000, 
    tb_log_name=datetime.now().strftime("%Y%m%d_%H%M")
)

# 4. save
model.save(os.path.join(final_model_dir, "sac_model"))  # save：模型参数和结构
env.save(os.path.join(final_model_dir, "vec_normalize.pkl"))  # !: save VecNormalize 过程中的归一化参数(均值、方差等)，以便后续评估或继续训练时使用

#model.save(model_path)
#env.save(f"{model_path}_vec_normalize.pkl")

print(f"训练完成，模型已保存")

argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
训练完成，模型已保存


### experimentF0

In [8]:
import os
from datetime import datetime

import gymnasium as gym
import panda_gym

from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback


# 1. 定义 save 路径
exp_name = "exp_F0_SAC_random"
base_path = f"./outputs/{exp_name}"

best_model_dir = os.path.join(base_path, "best_model")
final_model_dir = os.path.join(base_path, "final_model")
log_dir = os.path.join(base_path, "logs")
os.makedirs(best_model_dir, exist_ok=True)
os.makedirs(final_model_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

final_model_path = os.path.join(final_model_dir, "final_model")
final_vec_path = os.path.join(final_model_dir, "final_model_vec_normalize.pkl")

# 2. 定义并包装环境
env_id = "PandaReach-v3"

def make_env():
    # 使用 reward_type="dense" 确保快速收敛
    env = gym.make(env_id, reward_type="dense")
    return env

# 包装环境：向量化 + 归一化
env = DummyVecEnv([make_env])
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)

# 评估环境（不更新归一化参数）
eval_env = DummyVecEnv([make_env])
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, training=False)


# 3. 设置 Callbacks
# 评估回调：每 2000 步测试一次成功率（默认用奖励评估）
eval_callback = EvalCallback(
    eval_env, 
    best_model_save_path= best_model_dir,   # best_model
    log_path=base_path,      # evaluations.npz
    eval_freq=2000,
    n_eval_episodes=20,       # 增加到 20 次，成功率才比较有参考价值
    deterministic=True,
    warn=False                # 减少不必要的警告
)

# 自动保存归一化的统计量（这是解决加载模型后变傻的关键）
class SaveVecNormalizeCallback(BaseCallback):
    def __init__(self, save_path: str, verbose=1):
        super(SaveVecNormalizeCallback, self).__init__(verbose)
        self.save_path = save_path
    def _on_step(self) -> bool:
        if self.n_calls % 5000 == 0:
            self.model.get_vec_normalize_env().save(self.save_path)
        return True
    
save_vec_callback = SaveVecNormalizeCallback(save_path=os.path.join(log_dir, "vec_normalize.pkl"))

# 4. 定义 SAC 算法
model = SAC(
    "MultiInputPolicy", 
    env,
    learning_rate=1e-3,
    buffer_size=100000,
    batch_size=256,
    tau=0.005,
    gamma=0.99,
    tensorboard_log=log_dir,
    verbose=0,   # 关日志输出
    device="cpu"  # 对于简单的 MLP，Mac 的 CPU 通常比 GPU 效率更高
)

# --- 5. 开始训练 ---
print("开始训练 PandaReach 任务...")
model.learn(
    total_timesteps=50000, 
    callback=[eval_callback, save_vec_callback],
    tb_log_name=datetime.now().strftime("%Y%m%d_%H%M")
)

# --- 6. 最终保存 ---
model.save(final_model_path)
env.save(final_vec_path)
print(f"训练完成！模型已保存至 {base_path}")

argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
开始训练 PandaReach 任务...
Eval num_timesteps=2000, episode_reward=-16.21 +/- 8.51
Episode length: 42.85 +/- 17.03
Success rate: 15.00%
New best mean reward!
Eval num_timesteps=4000, episode_reward=-14.74 +/- 5.85
Episode length: 50.00 +/- 0.00
Success rate: 0.00%
New best mean reward!
Eval num_timesteps=6000, episode_reward=-13.17 +/- 5.34
Episode length: 50.00 +/- 0.00
Success rate: 0.00%
New best mean reward!
Eval num_timesteps=8000, episode_reward=-11.18 +/- 4.63
Episode length: 49.30 +/- 3.05
Success rate: 5.00%
New best mean reward!
Eval num_timesteps=10000, episode_reward=-10.26 +/- 3.37
Episode length: 50.00 +/- 0.00
Success rate: 0.00%
New best mean reward!
Eval num_timesteps=120

> TODO: 
1. final+断点续训逻辑
2. best 需同步保存 pkl
3. best 选择逻辑，成功率？
4. SaveVecNormalizeCallback需要重新定义，原来的无意义

In [ ]:
# --- 2. 自定义 Callback：确保最佳模型拥有配套的 pkl ---
class SaveCallback(BaseCallback):
    def __init__(self, save_path: str, verbose=1):
        super(SaveCallback, self).__init__(verbose)
        self.save_path = save_path
        self.last_best_reward = -float('inf')

    def _on_step(self) -> bool:
        # 检查 EvalCallback 是否找到了新的最佳模型
        # EvalCallback 会在 self.parent.best_mean_reward 中存储最高分
        if hasattr(self.parent, 'best_mean_reward'):
            current_best_reward = self.parent.best_mean_reward
            if current_best_reward > self.last_best_reward:
                self.last_best_reward = current_best_reward
                if self.verbose > 0:
                    print(f"检测到新最佳奖励: {current_best_reward:.2f}，保存对应归一化统计量...")
                # 保存与当前最佳模型匹配的 pkl
                self.model.get_vec_normalize_env().save(
                    os.path.join(self.save_path, "best_model_vec_normalize.pkl")
                )
        return True

In [ ]:
import os
import gymnasium as gym
import panda_gym
from datetime import datetime
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback, CheckpointCallback, CallbackList

# --- 1. 路径配置 ---
exp_name = "exp_F2_SAC_random"
base_path = f"./outputs/{exp_name}"

best_model_dir = os.path.join(base_path, "best_model")
final_model_dir = os.path.join(base_path, "final_model")
log_dir = os.path.join(base_path, "logs")

os.makedirs(best_model_dir, exist_ok=True)
os.makedirs(final_model_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

# 具体的模型与统计量文件路径
final_model_path = os.path.join(final_model_dir, "final_model.zip")
final_vec_path = os.path.join(final_model_dir, "final_model_vec_normalize.pkl")

# --- 2. 自定义 Callback 逻辑 ---

class BestModelSyncCallback(BaseCallback):
    """当 EvalCallback 发现最佳模型时，同步保存 VecNormalize 统计量"""
    def __init__(self, train_env: VecNormalize, save_path: str, verbose=1):
        super().__init__(verbose)
        self.train_env = train_env
        self.save_path = save_path

    def _on_step(self) -> bool:
        # 这个方法会在被 EvalCallback 的 callback_on_new_best 触发时执行
        save_path_full = os.path.join(self.save_path, "best_model_vec_normalize.pkl")
        self.train_env.save(save_path_full)
        if self.verbose > 0:
            print(f"检测到新最佳得分！同步保存归一化统计量至: {save_path_full}")
        return True

class SyncEvalStatsCallback(BaseCallback):
    """在评估之前，确保 eval_env 使用 train_env 的均值和方差"""
    def __init__(self, train_env: VecNormalize, eval_env: VecNormalize, verbose=0):
        super().__init__(verbose)
        self.train_env = train_env
        self.eval_env = eval_env

    def _on_step(self) -> bool:
        # 强制同步 observation 统计量（评估时不使用 reward 归一化，所以只同步 obs）
        self.eval_env.obs_rms = self.train_env.obs_rms
        return True

# --- 3. 环境准备函数 ---
env_id = "PandaReach-v3"

def make_env():
    return gym.make(env_id, reward_type="dense")

# --- 4. 实例化环境与加载逻辑 ---

# 检查是否存在已有的模型以续传
is_resume = os.path.exists(final_model_path) and os.path.exists(final_vec_path)

# 创建训练环境
train_env = DummyVecEnv([make_env])
if is_resume:
    print(f"检测到历史存档，正在加载归一化统计量: {final_vec_path}")
    train_env = VecNormalize.load(final_vec_path, train_env)
else:
    train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True)

# 创建评估环境 (评估环境不需要 reward 归一化)
eval_env = DummyVecEnv([make_env])
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, training=False)

# --- 5. 构建模型 ---

if is_resume:
    print(f"正在从存档加载模型: {final_model_path}")
    model = SAC.load(
        final_model_path, 
        env=train_env, 
        device="cpu", # 或 "cuda"
        tensorboard_log=log_dir
    )
else:
    print("未发现存档，开始全新训练。")
    model = SAC(
        "MultiInputPolicy", 
        train_env,
        tensorboard_log=log_dir,
        verbose=1,
        device="cpu"
    )

# --- 6. 设置 Callbacks 组合 ---

# 1. 发现最佳模型时的保存动作
on_best_callback = BestModelSyncCallback(train_env=train_env, save_path=best_model_dir)

# 2. 评估前的统计量同步
sync_eval_callback = SyncEvalStatsCallback(train_env=train_env, eval_env=eval_env)

# 3. 核心评估回调
eval_callback = EvalCallback(
    eval_env, 
    best_model_save_path=best_model_dir, 
    log_path=base_path,
    eval_freq=2000,
    deterministic=True,
    verbose=1,
    callback_on_new_best=on_best_callback, # 发现最佳时触发保存 pkl
    callback_on_step=sync_eval_callback    # 每次评估前同步统计量
)

# 4. 定期检查点回调 (用于意外中断后恢复)
checkpoint_callback = CheckpointCallback(
    save_freq=10000, 
    save_path=final_model_dir, 
    name_prefix="final_model", # 覆盖或滚动保存
    save_vecnormalize=True
)

# 组合所有 Callback
callback_list = CallbackList([eval_callback, checkpoint_callback])

# --- 7. 训练 ---
total_steps = 50000
print(f"训练开始，文件将保存至: {base_path}")

try:
    model.learn(
        total_timesteps=total_steps, 
        callback=callback_list,
        tb_log_name=datetime.now().strftime("%Y%m%d_%H%M"),
        reset_num_timesteps=not is_resume # 如果是续传，保持时间步计数不重置
    )
except KeyboardInterrupt:
    print("训练被手动中断，正在保存当前进度...")

# --- 8. 最终保存 ---
# 保存最终模型 (用于续传)
model.save(final_model_path)
# 保存最终归一化统计量 (用于续传)
train_env.save(final_vec_path)

print(f"训练任务完成！最终模型已存至: {final_model_dir}")